# Libraries

In [34]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns


from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

In [ ]:
# dispongo la carpeta con funciones e importo ternara. 
from pathlib import Path
import sys
import importlib


# Obtiene la ruta del directorio actual y sube un nivel (.parent)
raiz_proyecto = Path().resolve().parent

# Agrega la ruta al sistema si no está ya incluida
if str(raiz_proyecto) not in sys.path:
  sys.path.append(str(raiz_proyecto))

# Importa tu librería o módulo
from funciones.ternaria import ternaria
from funciones.graficos_clusters_pdf  import generar_pdf_clusters


# URLs y Constantes

In [30]:
BASE_URL = os.path.join('/mnt/', 'c')
BASE_DATA_URL = os.path.join('/mnt/', 'e')
COMP_URL = os.path.join(BASE_URL, 'Users', 'marco', 'Desktop', 'MASTER', 'MASTER', 'DMEyF', 'COMP1')
DATA_FOLDER = os.path.join(BASE_DATA_URL, 'DATASETS', 'DMEyF')
DATA_URL = os.path.join(DATA_FOLDER, 'competencia_01_crudo.csv')
DATA_TERNARIA_URL = os.path.join(DATA_FOLDER, 'competencia_01_ternaria.csv')
DATA_DICT_URL= os.path.join(DATA_FOLDER, 'data_dict.csv')

SEED = 230047

In [29]:
os.listdir(DATA_FOLDER)

['competencia_01_crudo.csv',
 'competencia_01_ternaria.csv',
 'data_dict.csv',
 'DiccionarioDatos_2026.ods']

# UTILS

In [4]:
def van_dongen_normalized(contingency):
    n = contingency.values.sum()
    sum_max_rows = contingency.max(axis=1).sum()  # para cada label, max sobre clusters
    sum_max_cols = contingency.max(axis=0).sum()  # para cada cluster, max sobre labels
    max_row_total = contingency.sum(axis=1).max()
    max_col_total = contingency.sum(axis=0).max()
    
    vdn = (2*n - sum_max_rows - sum_max_cols) / (2*n - max_row_total - max_col_total)
    return vdn

# LEEMOS DATOS

In [5]:
df = pd.read_csv(
    DATA_TERNARIA_URL,
    dtype={'numero_de_cliente': 'int32', 'foto_mes': 'int32'}
)

/tmp/ipykernel_202073/3175415450.py:1: DtypeWarning: Columns (0: ternaria) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


# ANALISIS

## Consolidamos columnas para primer analisis

In [6]:
def agregar_mes_baja(
    df,
    cliente_col="numero_de_cliente",
    mes_col="foto_mes"
):
    resultado = df.copy()

    ultimo_mes_dataset = resultado[mes_col].max()

    ultimo_mes_cliente = (
        resultado
        .groupby(cliente_col)[mes_col]
        .max()
    )

    mes_baja = pd.Series(
        pd.NA,
        index=ultimo_mes_cliente.index,
        dtype="Int64"
    )

    clientes_baja = ultimo_mes_cliente < ultimo_mes_dataset
    ultimo_mes = ultimo_mes_cliente.loc[clientes_baja]

    # Calcula correctamente el mes siguiente, incluyendo diciembre → enero
    mes_baja.loc[clientes_baja] = (
        (ultimo_mes // 100 + (ultimo_mes % 100 == 12)) * 100
        + (ultimo_mes % 100) % 12 + 1
    ).astype("Int64")

    resultado["mes_baja"] = (
        resultado[cliente_col]
        .map(mes_baja)
        .astype("Int64")
    )

    return resultado

In [7]:
# Agregamos columna con mes q se da de baja quien asi lo haya hecho.
df = agregar_mes_baja(df)

In [8]:
diccionario_consolidaciones = {
    "mtarjetas_consumo": {
        "columnas": [
            "mautoservicio",
            "mtarjeta_visa_consumo",
            "mtarjeta_master_consumo"
        ],
        "operacion": "sum"
    },
    "mtarjetas_pagado": {
        "columnas": [
            "Visa_mpagado",
            "Master_mpagado"
        ],
        "operacion": "sum"
    },
    "msueldo_acreditado": {
        "columnas": [
            "mpayroll",
            "mpayroll2"
        ],
        "operacion": "sum"
    }
}
columnas_base = [
        "numero_de_cliente",
        "foto_mes",
        "mes_baja",
        "active_quarter",
        "cliente_vip",

        # Actividad general y digital
        "ctrx_quarter",
        "chomebanking_transacciones",
        "cmobile_app_trx",

        # Transferencias
        "mtransferencias_recibidas",
        "mtransferencias_emitidas",

        # Pagos habituales
        "mpagodeservicios",
        "mpagomiscuentas",
        "mcuenta_debitos_automaticos",

        # Extracciones
        "mextraccion_autoservicio",

        # Mantenimiento, saldo y productos
        "mcomisiones_mantenimiento",
        "mcuentas_saldo",
        "cproductos"
    ]

In [9]:
def consolidar_columnas(
    df,
    columnas_base,
    diccionario_consolidaciones
):

    df_salida = df[columnas_base].copy()

    for columna_nueva, configuracion in diccionario_consolidaciones.items():

        columnas_origen = configuracion["columnas"]
        operacion = configuracion["operacion"].lower()

        if operacion == "sum":
            df_salida[columna_nueva] = df[columnas_origen].sum(
                axis=1,
                min_count=1
            )

        elif operacion == "avg":
            df_salida[columna_nueva] = df[columnas_origen].mean(axis=1)

        elif operacion == "max":
            df_salida[columna_nueva] = df[columnas_origen].max(axis=1)

        elif operacion == "min":
            df_salida[columna_nueva] = df[columnas_origen].min(axis=1)

        else:
            raise ValueError(
                f"Operación no válida para {columna_nueva}: {operacion}"
            )

    df_salida["ternaria"] = df["ternaria"]

    return df_salida

In [10]:
df_resumen = consolidar_columnas(df,columnas_base, diccionario_consolidaciones)

In [11]:
df_resumen.columns

Index(['numero_de_cliente', 'foto_mes', 'mes_baja', 'active_quarter',
       'cliente_vip', 'ctrx_quarter', 'chomebanking_transacciones',
       'cmobile_app_trx', 'mtransferencias_recibidas',
       'mtransferencias_emitidas', 'mpagodeservicios', 'mpagomiscuentas',
       'mcuenta_debitos_automaticos', 'mextraccion_autoservicio',
       'mcomisiones_mantenimiento', 'mcuentas_saldo', 'cproductos',
       'mtarjetas_consumo', 'mtarjetas_pagado', 'msueldo_acreditado',
       'ternaria'],
      dtype='str')

In [12]:
mask_baja = df_resumen['ternaria'].str.contains('BAJA') 

In [13]:
proporcion_ceros = (
    df_resumen.loc[mask_baja].select_dtypes(include="number")
      .eq(0)
      .sum()
      .div(len(df_resumen.loc[mask_baja]))
      .sort_values(ascending=False)
)
proporcion_ceros

cliente_vip                    0.999237
mpagodeservicios               0.998255
msueldo_acreditado             0.936859
mextraccion_autoservicio       0.864449
mpagomiscuentas                0.861941
mcuenta_debitos_automaticos    0.826609
mtransferencias_emitidas       0.770011
mtarjetas_pagado               0.708506
mtransferencias_recibidas      0.574809
cmobile_app_trx                0.543184
mtarjetas_consumo              0.476772
chomebanking_transacciones     0.452236
mcomisiones_mantenimiento      0.327808
active_quarter                 0.148419
ctrx_quarter                     0.1241
mcuentas_saldo                   0.0241
mes_baja                            0.0
foto_mes                            0.0
numero_de_cliente                   0.0
cproductos                          0.0
dtype: Float64

# K-Means

In [14]:
# columnas que no tienen valor informativo
columnas_excluir = [
    "numero_de_cliente",
    "foto_mes",
    "ternaria"
]
# Siempre agrego n registros de no baja para ver si el clustering los agrupa separado de los demas. 
def get_random_continua(k, mask_baja):
    indices_random_continua = (
        df.loc[df["ternaria"] == "continua"]
        .sample(
            n=round(mask_baja.sum() / k),
            random_state=SEED
        )
        .index)
    mask_random_continua = df.index.isin(indices_random_continua)
    return mask_random_continua

In [15]:
# columnas que no tienen valor informativo

sparsity_rates = [0.2, 0.5, 0.8]
ks = [3,4,5,6]
resultados = []
dfs_baja = {}
for rate in sparsity_rates:
    sparse_columns = []
    non_sparse_columns = []  
    for columna in proporcion_ceros.items():
        if columna[1]>rate:
            sparse_columns.append(columna[0])
        else :
            non_sparse_columns.append(columna[0])
    _columnas_excluir = columnas_excluir + [x for x in sparse_columns]
    for k in ks:
        # agrego registros q no son de baja, quiero ver si el clustering los distingue
        mask_random = get_random_continua(k,mask_baja)
        df_baja = df_resumen.loc[mask_baja|mask_random].copy()
        df_baja.fillna(0, inplace=True) #naively filll with 0s
        labels_reales = df_baja['ternaria']
        X = df_baja.drop(columns = _columnas_excluir).select_dtypes(include='number')
        X_escalado = StandardScaler().fit_transform(X)

        kmeans = KMeans(
        n_clusters=k+1,
        n_init=20,
        random_state=SEED
        )
        column_name = f'k-mean-{k}-spars-{rate*10:.0f}'
        df_baja[column_name] = kmeans.fit_predict(X_escalado)
        dfs_baja[(rate, k)] = df_baja.copy()
        labels = kmeans.labels_
        resultados.append({
            'column_name':column_name,
            "sparsity_rate": rate,
            "k_baja": k,
            "clusters_totales": k + 1,
            "tamanios_clusters": pd.Series(labels).value_counts().sort_index().to_dict(),
            "columnas_usadas": X.shape[1],
            "columnas_excluidas_sparse": len(sparse_columns),
            "inercia": kmeans.inertia_,                                                   # menor
            "iteraciones": kmeans.n_iter_,
            "silhouette": silhouette_score(X_escalado, labels),                          # mayor
            "davies_bouldin": davies_bouldin_score(X_escalado, labels),                   # menor
            "calinski_harabasz": calinski_harabasz_score(X_escalado, labels)              # mayor
        })
tabla_resultados = pd.DataFrame(resultados)

tabla_resultados.sort_values(
    "silhouette",
    ascending=False
)

,column_name,sparsity_rate,k_baja,clusters_totales,tamanios_clusters,columnas_usadas,columnas_excluidas_sparse,inercia,iteraciones,silhouette,davies_bouldin,calinski_harabasz
0,k-mean-3-spars-2,0.2,3,4,"{0: 7645, 1: 3177, 2: 1387, 3: 18}",5,13,22575.293225,5,0.538350,0.696323,6959.160932
3,k-mean-6-spars-2,0.2,6,7,"{0: 877, 1: 1373, 2: 385, 3: 3107, 4: 10, 5: 3...",5,13,11316.298976,12,0.474850,0.814425,6640.560973
2,k-mean-5-spars-2,0.2,5,6,"{0: 3929, 1: 1377, 2: 498, 3: 10, 4: 1513, 5: ...",5,13,13595.369159,23,0.450753,0.882644,6702.107637
1,k-mean-4-spars-2,0.2,4,5,"{0: 2300, 1: 3870, 2: 15, 3: 3896, 4: 1381}",5,13,16991.210065,12,0.433701,0.755181,6796.638967
5,k-mean-4-spars-5,0.5,4,5,"{0: 1822, 1: 7338, 2: 15, 3: 1381, 4: 906}",8,10,44181.549432,41,0.390463,1.132672,3080.322720
4,k-mean-3-spars-5,0.5,3,4,"{0: 7524, 1: 3299, 2: 17, 3: 1387}",8,10,54399.993057,8,0.382381,0.990758,3251.678809
7,k-mean-6-spars-5,0.5,6,7,"{0: 1193, 1: 4704, 2: 1373, 3: 405, 4: 10, 5: ...",8,10,33077.395433,22,0.332547,1.189582,2828.457834
6,k-mean-5-spars-5,0.5,5,6,"{0: 4764, 1: 2645, 2: 1377, 3: 1472, 4: 736, 5...",8,10,37222.955806,13,0.324222,1.182617,3002.437857
8,k-mean-3-spars-8,0.8,3,4,"{0: 3189, 1: 7650, 2: 1387, 3: 1}",12,6,96700.675890,9,0.298964,1.047676,2107.663223
11,k-mean-6-spars-8,0.8,6,7,"{0: 1252, 1: 1373, 2: 645, 3: 3731, 4: 10, 5: ...",12,6,65849.990604,8,0.292039,1.392696,1691.890802


In [16]:
df_baja = dfs_baja[(0.2, 4)]
pd.crosstab(
        df_baja["k-mean-4-spars-2"],
        df_baja["ternaria"],
        margins=True
    )

ternaria,BAJA+1,BAJA+2,continua,All
k-mean-4-spars-2,,,,
0,30,27,2243,2300
1,2225,1634,11,3870
2,4,2,9,15
3,2053,1834,9,3896
4,791,570,20,1381
All,5103,4067,2292,11462


In [17]:
df_baja = dfs_baja[(0.8, 4)]
pd.crosstab(
        df_baja["k-mean-4-spars-8"],
        df_baja["ternaria"],
        margins=True
    )

ternaria,BAJA+1,BAJA+2,continua,All
k-mean-4-spars-8,,,,
0,1984,1691,245,3920
1,214,216,1814,2244
2,791,570,20,1381
3,7,5,18,30
4,2107,1585,195,3887
All,5103,4067,2292,11462


In [18]:
sparsity_rates = [0.2, 0.5, 0.8]
ks = [3,4,5,6]
resultados = []
# agrego registros q no son de baja, quiero ver si el clustering los distingue
mask_random = get_random_continua(5,mask_baja) #tomamos una proporcion siempre como si k fuera 5
df_baja = df_resumen.loc[mask_baja|mask_random].copy()
df_baja.fillna(0, inplace=True) #naively filll with 0s
labels_reales = df_baja['ternaria']
for rate in sparsity_rates:
    sparse_columns = []
    non_sparse_columns = []  
    for columna in proporcion_ceros.items():
        if columna[1]>rate:
            sparse_columns.append(columna[0])
        else :
            non_sparse_columns.append(columna[0])

    _columnas_excluir = columnas_excluir + [x for x in sparse_columns]

    X = df_baja.drop(columns = _columnas_excluir).select_dtypes(include='number')
    X_escalado = StandardScaler().fit_transform(X)

    for k in ks:

        kmeans = KMeans(
        n_clusters=k+1,
        n_init=20,
        random_state=SEED
        )
        column_name = f'k-mean-{k}-spars-{rate*10:.0f}'
        df_baja[column_name] = kmeans.fit_predict(X_escalado)

        labels = kmeans.labels_
        resultados.append({
            'column_name':column_name,
            "sparsity_rate": rate,
            "k_baja": k,
            "clusters_totales": k + 1,
            "tamanios_clusters": pd.Series(labels).value_counts().sort_index().to_dict(),
            "ncolumnas_usadas": X.shape[1],
            "columnas_usadas": non_sparse_columns,
            "columnas_excluidas_sparse": len(sparse_columns),
            "inercia": kmeans.inertia_,                                                   # menor
            "iteraciones": kmeans.n_iter_,
            "silhouette": silhouette_score(X_escalado, labels),                          # mayor
            "davies_bouldin": davies_bouldin_score(X_escalado, labels),                   # menor
            "calinski_harabasz": calinski_harabasz_score(X_escalado, labels)              # mayor
        })
tabla_resultados = pd.DataFrame(resultados)

tabla_resultados.sort_values(
    "silhouette",
    ascending=False
)

,column_name,sparsity_rate,k_baja,clusters_totales,tamanios_clusters,ncolumnas_usadas,columnas_usadas,columnas_excluidas_sparse,inercia,iteraciones,silhouette,davies_bouldin,calinski_harabasz
0,k-mean-3-spars-2,0.2,3,4,"{0: 1942, 1: 10, 2: 1377, 3: 7675}",5,"[active_quarter, ctrx_quarter, mcuentas_saldo,...",13,20707.432544,12,0.560277,0.651653,6075.731732
7,k-mean-6-spars-5,0.5,6,7,"{0: 3104, 1: 3843, 2: 1377, 3: 1499, 4: 402, 5...",12,"[mtarjetas_consumo, chomebanking_transacciones...",10,38097.649922,7,0.527506,1.031998,4519.841346
6,k-mean-5-spars-5,0.5,5,6,"{0: 3864, 1: 3098, 2: 1836, 3: 819, 4: 1377, 5...",12,"[mtarjetas_consumo, chomebanking_transacciones...",10,43115.190082,5,0.522291,0.967094,4537.069379
10,k-mean-5-spars-8,0.8,5,6,"{0: 3862, 1: 3102, 2: 1377, 3: 32, 4: 1827, 5:...",20,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,84288.848681,10,0.521973,1.409466,3543.602993
11,k-mean-6-spars-8,0.8,6,7,"{0: 3103, 1: 3862, 2: 8, 3: 1832, 4: 812, 5: 1...",20,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,78081.212699,3,0.520348,0.930896,3333.197599
5,k-mean-4-spars-5,0.5,4,5,"{0: 3864, 1: 3098, 2: 1846, 3: 1377, 4: 819}",12,"[mtarjetas_consumo, chomebanking_transacciones...",10,50491.172087,5,0.517713,1.095387,4441.586089
9,k-mean-4-spars-8,0.8,4,5,"{0: 817, 1: 3863, 2: 3103, 3: 1844, 4: 1377}",20,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,92771.260502,4,0.513809,1.084441,3773.444540
4,k-mean-3-spars-5,0.5,3,4,"{0: 1377, 1: 3728, 2: 1966, 3: 3933}",12,"[mtarjetas_consumo, chomebanking_transacciones...",10,57977.704882,5,0.486863,0.926050,4684.405543
8,k-mean-3-spars-8,0.8,3,4,"{0: 1377, 1: 3104, 2: 3863, 3: 2660}",20,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,106793.917147,5,0.475798,1.092045,3889.568946
3,k-mean-6-spars-2,0.2,6,7,"{0: 851, 1: 3796, 2: 1377, 3: 468, 4: 1382, 5:...",5,"[active_quarter, ctrx_quarter, mcuentas_saldo,...",13,11748.617305,17,0.472670,0.818989,6750.572245


In [19]:
df_baja

,numero_de_cliente,foto_mes,mes_baja,active_quarter,cliente_vip,ctrx_quarter,chomebanking_transacciones,cmobile_app_trx,mtransferencias_recibidas,mtransferencias_emitidas,...,k-mean-5-spars-2,k-mean-6-spars-2,k-mean-3-spars-5,k-mean-4-spars-5,k-mean-5-spars-5,k-mean-6-spars-5,k-mean-3-spars-8,k-mean-4-spars-8,k-mean-5-spars-8,k-mean-6-spars-8
124,12188338,202103,202105,1,0,80,8,1,0.00,0.00,...,5,6,1,1,1,0,1,2,1,0
138,12191939,202103,0,1,0,87,21,1,27483.97,21382.39,...,4,4,2,2,2,3,3,3,4,3
183,12200827,202103,202105,1,0,23,45,0,938.40,0.00,...,0,1,3,0,0,1,2,1,0,1
249,12218654,202103,202104,1,0,5,0,1,0.00,0.00,...,5,6,1,1,1,0,1,2,1,0
288,12227617,202103,202105,1,0,8,0,0,0.00,0.00,...,0,1,3,0,0,1,2,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818040,77515244,202107,202108,1,0,11,0,1,0.00,0.00,...,5,6,1,1,1,0,1,2,1,0
818111,77582551,202107,202108,1,0,3,0,1,0.00,0.00,...,5,6,1,1,1,0,1,2,1,0
818208,77733822,202107,202108,0,0,2,0,0,0.00,0.00,...,1,2,0,3,4,2,0,4,2,5
818403,78074138,202107,202108,0,0,2,1,1,0.00,0.00,...,1,2,0,3,4,2,0,4,2,5


In [20]:
for column in tabla_resultados['column_name'].unique():
    print( pd.crosstab(
        df_baja[column],
        df_baja["ternaria"],
        margins=True
    ))

ternaria          BAJA+1  BAJA+2  continua    All
k-mean-3-spars-2                                 
0                     70      72      1800   1942
1                      3       1         6     10
2                    791     570        16   1377
3                   4239    3424        12   7675
All                 5103    4067      1834  11004
ternaria          BAJA+1  BAJA+2  continua    All
k-mean-4-spars-2                                 
0                   2215    1626         9   3850
1                     30      24      1796   1850
2                      3       1         6     10
3                    791     570        16   1377
4                   2064    1846         7   3917
All                 5103    4067      1834  11004
ternaria          BAJA+1  BAJA+2  continua    All
k-mean-5-spars-2                                 
0                   2263    1661         5   3929
1                    791     570        16   1377
2                     80      97       321    498


# Hierarchical


In [21]:
sparsity_rates = [0.2, 0.5, 0.8]
ks = [3,4,5,6]
linkages = ["ward", "complete", "average", "single"]
metrics = ["euclidean", "manhattan", "cosine"]

resultados = []

# agrego registros q no son de baja, quiero ver si el clustering los distingue
mask_random = get_random_continua(5,mask_baja) #tomamos una proporcion siempre como si k fuera 5
df_baja = df_resumen.loc[mask_baja|mask_random].copy()
df_baja.fillna(0, inplace=True) #naively filll with 0s
labels_reales = df_baja['ternaria']

for rate in sparsity_rates:
    sparse_columns = []
    non_sparse_columns = []

    for columna in proporcion_ceros.items():
        if columna[1] > rate:
            sparse_columns.append(columna[0])
        else:
            non_sparse_columns.append(columna[0])

    _columnas_excluir = columnas_excluir + [x for x in sparse_columns]

    for k in ks:
        X = (
            df_baja
            .drop(columns=_columnas_excluir)
            .select_dtypes(include="number")
        )

        X_escalado = StandardScaler().fit_transform(X)

        for linkage in linkages:
            for metric in metrics:

                # Ward solamente admite distancia euclídea
                if linkage == "ward" and metric != "euclidean":
                    continue

                modelo = AgglomerativeClustering(
                    n_clusters=k + 1,
                    linkage=linkage,
                    metric=metric
                )

                labels = modelo.fit_predict(X_escalado)

                column_name = (
                    f"agg-{k}-{linkage}-{metric}-spars-{rate*10:.0f}"
                )

                df_baja[column_name] = labels

                resultados.append({
                    "column_name": column_name,
                    "sparsity_rate": rate,
                    "k_baja": k,
                    "clusters_totales": k + 1,
                    "linkage": linkage,
                    "metric": metric,
                    "columnas_usadas": X.shape[1],
                    "columnas_usadas": non_sparse_columns,
                    "columnas_excluidas_sparse": len(sparse_columns),
                    "silhouette": silhouette_score(X_escalado, labels),
                    "davies_bouldin": davies_bouldin_score(
                        X_escalado,
                        labels
                    ),
                    "calinski_harabasz": calinski_harabasz_score(
                        X_escalado,
                        labels
                    ),
                    "tamanios_clusters": (
                        pd.Series(labels)
                        .value_counts()
                        .sort_index()
                        .to_dict()
                    )
                })

tabla_resultados = pd.DataFrame(resultados)

/tmp/ipykernel_202073/4014153423.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_baja[column_name] = labels
/tmp/ipykernel_202073/4014153423.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_baja[column_name] = labels
/tmp/ipykernel_202073/4014153423.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe =

In [22]:
tabla_resultados.sort_values(by='silhouette', ascending=False)

,column_name,sparsity_rate,k_baja,clusters_totales,linkage,metric,columnas_usadas,columnas_excluidas_sparse,silhouette,davies_bouldin,calinski_harabasz,tamanios_clusters
81,agg-3-complete-euclidean-spars-8,0.8,3,4,complete,euclidean,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,0.958023,0.292596,1494.265463,"{0: 10999, 1: 2, 2: 2, 3: 1}"
85,agg-3-average-manhattan-spars-8,0.8,3,4,average,manhattan,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,0.958023,0.292596,1494.265463,"{0: 10999, 1: 2, 2: 1, 3: 2}"
88,agg-3-single-manhattan-spars-8,0.8,3,4,single,manhattan,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,0.958023,0.292596,1494.265463,"{0: 10999, 1: 2, 2: 2, 3: 1}"
84,agg-3-average-euclidean-spars-8,0.8,3,4,average,euclidean,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,0.958023,0.292596,1494.265463,"{0: 10999, 1: 2, 2: 1, 3: 2}"
82,agg-3-complete-manhattan-spars-8,0.8,3,4,complete,manhattan,"[mtransferencias_emitidas, mtarjetas_pagado, m...",6,0.958023,0.292596,1494.265463,"{0: 10999, 1: 2, 2: 2, 3: 1}"
...,...,...,...,...,...,...,...,...,...,...,...,...
49,agg-3-single-cosine-spars-5,0.5,3,4,single,cosine,"[mtarjetas_consumo, chomebanking_transacciones...",10,0.450544,0.891509,2441.642307,"{0: 19, 1: 9605, 2: 1378, 3: 2}"
6,agg-3-average-cosine-spars-2,0.2,3,4,average,cosine,"[active_quarter, ctrx_quarter, mcuentas_saldo,...",13,0.427065,0.884402,4728.179625,"{0: 3678, 1: 4111, 2: 1377, 3: 1838}"
63,agg-5-complete-cosine-spars-5,0.5,5,6,complete,cosine,"[mtarjetas_consumo, chomebanking_transacciones...",10,0.416777,1.236123,953.773392,"{0: 2474, 1: 183, 2: 2530, 3: 1734, 4: 2877, 5..."
3,agg-3-complete-cosine-spars-2,0.2,3,4,complete,cosine,"[active_quarter, ctrx_quarter, mcuentas_saldo,...",13,0.400005,0.996357,4047.050332,"{0: 2798, 1: 3769, 2: 1377, 3: 3060}"


In [23]:
df_baja

,numero_de_cliente,foto_mes,mes_baja,active_quarter,cliente_vip,ctrx_quarter,chomebanking_transacciones,cmobile_app_trx,mtransferencias_recibidas,mtransferencias_emitidas,...,agg-6-ward-euclidean-spars-8,agg-6-complete-euclidean-spars-8,agg-6-complete-manhattan-spars-8,agg-6-complete-cosine-spars-8,agg-6-average-euclidean-spars-8,agg-6-average-manhattan-spars-8,agg-6-average-cosine-spars-8,agg-6-single-euclidean-spars-8,agg-6-single-manhattan-spars-8,agg-6-single-cosine-spars-8
124,12188338,202103,202105,1,0,80,8,1,0.00,0.00,...,6,0,0,5,1,0,3,2,2,0
138,12191939,202103,0,1,0,87,21,1,27483.97,21382.39,...,1,0,0,1,1,0,2,2,2,0
183,12200827,202103,202105,1,0,23,45,0,938.40,0.00,...,4,0,0,0,1,0,4,2,2,0
249,12218654,202103,202104,1,0,5,0,1,0.00,0.00,...,1,0,0,1,1,0,2,2,2,0
288,12227617,202103,202105,1,0,8,0,0,0.00,0.00,...,4,0,0,0,1,0,4,2,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818040,77515244,202107,202108,1,0,11,0,1,0.00,0.00,...,6,0,0,5,1,0,3,2,2,0
818111,77582551,202107,202108,1,0,3,0,1,0.00,0.00,...,6,0,0,5,1,0,3,2,2,0
818208,77733822,202107,202108,0,0,2,0,0,0.00,0.00,...,3,0,0,3,1,0,5,2,2,1
818403,78074138,202107,202108,0,0,2,1,1,0.00,0.00,...,3,0,0,3,1,0,5,2,2,1


In [24]:
pd.crosstab(
        df_baja["agg-5-complete-cosine-spars-5"],
        df_baja["ternaria"],
        margins=True
    )

ternaria,BAJA+1,BAJA+2,continua,All
agg-5-complete-cosine-spars-5,,,,
0,1315,1034,125,2474
1,104,78,1,183
2,1489,1036,5,2530
3,21,15,1698,1734
4,1513,1359,5,2877
5,661,545,0,1206
All,5103,4067,1834,11004


In [25]:
pd.crosstab(
        df_baja["agg-3-complete-cosine-spars-2"],
        df_baja["ternaria"],
        margins=True
    )

ternaria,BAJA+1,BAJA+2,continua,All
agg-3-complete-cosine-spars-2,,,,
0,523,467,1808,2798
1,2171,1593,5,3769
2,791,570,16,1377
3,1618,1437,5,3060
All,5103,4067,1834,11004


In [26]:
pd.crosstab(
        df_baja["agg-3-average-cosine-spars-2"],
        df_baja["ternaria"],
        margins=True
    )

ternaria,BAJA+1,BAJA+2,continua,All
agg-3-average-cosine-spars-2,,,,
0,2116,1551,11,3678
1,2174,1930,7,4111
2,791,570,16,1377
3,22,16,1800,1838
All,5103,4067,1834,11004


In [38]:
import importlib
import funciones.graficos_clusters_pdf as graficos_clusters_pdf

importlib.reload(graficos_clusters_pdf)

generar_pdf_clusters = graficos_clusters_pdf.generar_pdf_clusters

In [40]:
columna_cluster = 'agg-3-average-cosine-spars-2'
file_name = f'graficos-{columna_cluster}-sobre-consolidado-resumen.pdf'
output_file = os.path.join( os.getcwd(), 'graficos_clusters', file_name)
generar_pdf_clusters(
    df_variables=df_resumen,
    df_baja=df_baja,
    columna_cluster=columna_cluster,
    archivo_salida=output_file,
    banda="ic95",
    data_dict=DATA_DICT_URL,
    diccionario_consolidaciones=diccionario_consolidaciones
)

PosixPath('/home/marco/code/DMEyF/dmeyf2026/mis_pruebas/graficos_clusters/graficos-agg-3-average-cosine-spars-2-sobre-consolidado-resumen.pdf')